In [1]:

# clone repo

import os

PROJECT_ROOT = "/content/Project_Generative_AI_for_Data_Augmentation"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}

Cloning into '/content/Project_Generative_AI_for_Data_Augmentation'...
remote: Enumerating objects: 268, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 268 (delta 42), reused 43 (delta 25), pack-reused 190 (from 1)
Receiving objects: 100% (268/268), 8.41 MiB | 18.24 MiB/s, done.
Resolving deltas: 100% (133/133), done.
/content/Project_Generative_AI_for_Data_Augmentation


In [2]:
# =============================
# CONTROLLED VERBOSITY
# =============================

# Disable HF download progress bars BEFORE importing anything HF-related

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [3]:
# Dependency install
INSTALL_DEPS = True

if INSTALL_DEPS:
    !pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.0 MB/s eta 0:00:00


In [4]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

import importlib
import captioning
importlib.reload(captioning)
from captioning import run_captioning

In [5]:
# Setup

import torch
import logging
from torchvision.datasets import OxfordIIITPet
import numpy as np
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import random
from transformers.utils import logging as transformers_logging
from huggingface_hub.utils import logging as hf_logging


# Silence transformers & HF logs (keep only errors)
transformers_logging.set_verbosity_error()
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [6]:
# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [7]:
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

In [8]:
PROJECT_ROOT # it must be /content/Project_Generative_AI_for_Data_Augmentation

'/content/Project_Generative_AI_for_Data_Augmentation'

In [9]:
# control flags. REMEMBER TO USE THE OTHERS AS WELL IN THE NB!!!
RUN_CAPTIONING = True
RUN_TEXT_VARIATION = False
RUN_IMAGE_GENERATION = False
RUN_TRAINING = False

In [10]:
# dataset loading

dataset_train = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="trainval",
    download=True
)

dataset_test = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="test",
    download=True
)

print("Train size:", len(dataset_train))
print("Test size:", len(dataset_test))

100%|██████████| 792M/792M [00:37<00:00, 20.9MB/s]
100%|██████████| 19.2M/19.2M [00:02<00:00, 9.48MB/s]


Train size: 3680
Test size: 3669


In [11]:
# extract labels
labels = dataset_train._labels
indices = np.arange(len(dataset_train))

# perform stratified split
train_small_idx, _ = train_test_split(
    indices,
    train_size=0.30,
    stratify=labels,
    random_state=42
)

SPLIT_DIR = os.path.join(PROJECT_ROOT, "data", "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

np.save(os.path.join(SPLIT_DIR, "train_small_indices.npy"), train_small_idx)

In [12]:
# # run for ALL
# # create subset dataset from training set
dataset_train_small = Subset(dataset_train, train_small_idx)

In [13]:
# run for 10
dataset_train_small_10 = Subset(dataset_train, train_small_idx[:10])

# Captioning

In [14]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

In [15]:
if RUN_CAPTIONING:

  device = "cuda" if torch.cuda.is_available() else "cpu"

  processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

  model = Blip2ForConditionalGeneration.from_pretrained(
      "Salesforce/blip2-opt-2.7b",
      torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32 # to reduce GPU memory usage
  )

  model.to(device)
  model.eval()



  captions_dict = run_captioning(
      # dataset_train_small=dataset_train_small, # FOR ALL
      dataset_train_small=dataset_train_small_10, # FOR 10
      model=model,
      processor=processor,
      device=device,
      output_path=CAPTION_PATH
  )

  del model
  del processor
  clear_gpu()
  !nvidia-smi

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
100%|██████████| 10/10 [00:08<00:00,  1.18it/s]


Full caption generation completed.
GPU memory cleared.
Fri Feb 20 11:55:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             54W /  400W |     546MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disa

In [ ]:
'''
import nbformat
import os

def hard_clean_notebook(path):
    nb = nbformat.read(path, as_version=4)

    if "widgets" in nb.metadata:
        del nb.metadata["widgets"]

    for cell in nb.cells:
        if "widgets" in cell.get("metadata", {}):
            del cell["metadata"]["widgets"]

    nbformat.write(nb, path)
    print(f"Cleaned: {os.path.basename(path)}")


# 🔥 Walk entire project and clean every notebook
for root, _, files in os.walk(PROJECT_ROOT):
    for file in files:
        if file.endswith(".ipynb"):
            hard_clean_notebook(os.path.join(root, file))

print("All notebooks in project HARD cleaned.")
'''

# Text variation

## FLAN-T5-Large Model



In [16]:
import text_variation_flan_large
importlib.reload(text_variation_flan_large)
from text_variation_flan_large import run_text_variation as run_flan_large

In [18]:
MAX_ITEMS = 10

In [19]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

In [20]:
run_flan_large(
    caption_file=CAPTION_PATH,
    max_items=MAX_ITEMS
)

  0%|          | 0/10 [00:00<?, ?it/s]


Original: a pomeranian dog this dog is sitting on the bed
Generated: ['three-leggi', 'The dogs has taken shelter by taking photos, jumping into someone the dogs with some kind dogs of this puppy she got on that a very dandly friend by an owner so to the rest', 'Two pets lying naked. Some young child. He sit and stand outside his family, waiting when there should have started dog walk through the streets, after their food bowl is made of sugar dough (he']


 10%|█         | 1/10 [00:03<00:30,  3.39s/it]


Original: a pomeranian dog my dog is sitting on the bed
Generated: ['dog dog wearing we want his best day while not taking care her dogs nap with owner for pet care kashgarhian on line service app store by using my offer coupon free for ', 'dogs pear', 'A dog nexteer , pomelaid who can walk in city the day to come on stage. It was our favourite.Hea. It bes! We did so we named everything']

Original: a havanese dog is sitting on the tennis court
Generated: ['Some dogs playing down below from the sidewalk when I walk right below him because you can feel as cold today here ya don have so very cool sunglasses! Thank me or if no so it', '', 'there have many animals playing as it goes swimming into open river behind green field behind golf house ivan an tomas the red dogs he has made an advertisement above and also plays frisian pet']


 20%|██        | 2/10 [00:05<00:23,  2.92s/it]

...


Original: a saint bernard dog is standing in the snow
Generated: ["Some guys is talking. Two white dog and two others look surprised, waiting ahead like it is winter at snow camp but nothing happening it the dog bark' to them the owner takes quick advantage or even", 'one big big cute black bear that looks angry near many men', 'this shot goes towards west with many hills standing as yet snow falling up, on these sunny weathery autumn nights of christmas time inside and at that great temperature outside snow still to roll past at home for']

Original: a havanese dog is standing on a wooden staircase
Generated: ['some cats climb high aboard and rest while humans jump', 'a large furriable spanimeisan has been stuck with sticks or on fire all month!he can reach the summit before vanizing on land!!a large hanah', 'Someone was in one area and nobody helped and left and someone was staying despite two guard cars being out ahead doing this work inside several others homes too quickly in fron

 90%|█████████ | 9/10 [00:21<00:02,  2.26s/it]


Original: a havanese dog is sitting on the steps
Generated: ['A French-Hausine dog and two English male children wearing blaze brown fur pose down as they approach their white couch as children climb the upper table at Ikea stores below they occupy our shopping', 'An English male white can to get over to him dog steps from his home. A black american, American bullshadge with short fuzzy blonde long', 'Someone is wearing pamp pans with yellow shoes because an airport was just about an average city step from an indoor stadium as you look behind tdw into its terminal on step 2 at home']

Original: a persian cat is looking angry
Generated: ['happy pan to see someone wearing makeup and standing off an umbrella to work.niice shot for him because your body could suffer an ankle/toal problem.recall these five benefits to consider', 'someone says no for no attick the perriant to play games when we see there.dy-goblin looks really tired from work here by myself just like it before me too by', "

100%|██████████| 10/10 [00:24<00:00,  2.42s/it]


Original: a persian cat is looking at the camera with an angry expression
Generated: ['black gray c. and pink perd to cat lying dead face next and black.f cat holding her tongue wide opened by gagging its mouth up is holding itself awake like crazy and', 'Some men looking into blacke doors', 'there to keep your cats nice for as much']


In [21]:
clear_gpu()
!nvidia-smi

GPU memory cleared.
Fri Feb 20 11:57:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             52W /  400W |     548MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------

FLAN-T5-Large was evaluated as a candidate model for caption rewriting.

However, qualitative analysis showed several consistent issues:

- **poor semantic preservation:** generated output often drifted away from original meaning

- **hallucinations:** frequent introduction of unrelated object, people, or scenes

Due to these limitations, FLAN-T5-Large was deemed unsuitable for controlled data augmentation.

## FLAN-T5-XL Model

In [22]:
from text_variation_flan_xl import run_text_variation as run_flan_xl

run_flan_xl(
    caption_file=CAPTION_PATH,
    max_items=10
)

  0%|          | 0/10 [00:00<?, ?it/s]


Original: a pomeranian dog this dog is sitting on the bed
Generated: ['a pomeranian dog this pomeranian is sitting on the bed', 'a pomeranian dog this pomeranian dog is sitting on the bed', 'a pomeranian dog this pomeranian is sitting on the bed .']


 10%|█         | 1/10 [00:01<00:16,  1.89s/it]


Original: a pomeranian dog my dog is sitting on the bed
Generated: ['a pomeranian dog my pomeranian is sitting on the bed', 'a pomeranian dog my dog is sitting on the bed', 'a pomeranian dog my pomeranian dog is sitting on the bed']

Original: a havanese dog is sitting on the tennis court
Generated: ['A Havanese dog is sitting on a tennis court.', 'A Havanese dog sits on a tennis court.', 'A Havanese dog is sitting on the tennis court.']


 20%|██        | 2/10 [00:03<00:13,  1.67s/it]

...


Original: a saint bernard dog is standing in the snow
Generated: ['a saint bernard dog is standing in the snow', 'a saint bernard is standing in the snow', 'a saint bernard dog standing in the snow']

Original: a havanese dog is standing on a wooden staircase
Generated: ['A Havanese dog is standing on a wooden staircase.', 'A Havanese dog is standing on a wooden staircase', 'A Havanese is standing on a wooden staircase.']


 90%|█████████ | 9/10 [00:13<00:01,  1.38s/it]


Original: a havanese dog is sitting on the steps
Generated: ['A Havanese dog is sitting on the steps.', 'A Havanese dog sits on the steps.', 'A Havanese dog is sitting on the steps']

Original: a persian cat is looking angry
Generated: ['a persian cat looks angry', 'a persian cat is looking angry', 'a persian cat is angry']


100%|██████████| 10/10 [00:14<00:00,  1.48s/it]


Original: a persian cat is looking at the camera with an angry expression
Generated: ['a persian cat is looking at the camera with an angry expression', 'a persian cat is looking at the camera with an angry expression .', 'a persian cat is looking at the camera with an angry expression.']


In [23]:
clear_gpu()
!nvidia-smi

GPU memory cleared.
Fri Feb 20 11:58:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             52W /  400W |     548MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------

FLAN-T5-XL produced grammatically correct and semantically faithful rewrites.

However, the generated variations showed very low lexical diversity, often resulting in near-duplicate sentences with only minor wording or punctuation changes.

Since the goal of this stage is meaningful data augmentation, higher variation diversity was required.

## Mistral 7B Instruct Model

In [24]:
from text_variation_mistral import run_text_variation as run_mistral

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CAPTIONS_DIR = os.path.join(DATA_DIR, "captions")
TEXT_VARIATIONS_DIR = os.path.join(DATA_DIR, "text_variations")


os.makedirs(TEXT_VARIATIONS_DIR, exist_ok=True)

CAPTION_FILE = os.path.join(
    CAPTIONS_DIR,
    "captions_train_small_10.json"
)

TEXT_VARIATION_FILE = os.path.join(
    TEXT_VARIATIONS_DIR,
    "text_variations_train_small_10.json"
)

run_mistral(
    caption_file = CAPTION_FILE,
    output_file= TEXT_VARIATION_FILE,
    max_items=10
)

100%|██████████| 10/10 [00:45<00:00,  4.57s/it]

Mistral text variations saved.


In [25]:
clear_gpu()
!nvidia-smi

GPU memory cleared.
Fri Feb 20 12:00:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             52W /  400W |     556MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------

In [ ]:
from src.image_generation import (CaptionSelector, SyntheticImageGenerator)

# Caption Selection

In [ ]:
selector = CaptionSelector()

selected_data = {}

for idx, data in text_variations.items():

    class_name = data["class_name"]
    original_captions = data["original_captions"]
    generated_captions = data["generated_captions"]

    selected_generated = selector.select_top_captions(
        original_captions,
        generated_captions,
        top_k=2
    )

    selected_data[idx] = {
    "class_name": class_name,
    "original_captions": original_captions,
    "selected_generated_captions": selected_generated
}

# Image Generation

In [ ]:
generator = SyntheticImageGenerator()

metadata = generator.generate_images(
            selected_data = selected_data,
            output_dir = "data/synthetic/images",
            checkpoint_file = "data/synthetic/generation_checkpoint.json",
            batch_size=4
    )